In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!mkdir -p /content/drive/MyDrive/APNet/APNetCode
!ls /content/drive/MyDrive/APNet

APNetCode  APNetModel


In [3]:
!git clone https://github.com/huynhthanhphong231004IT/APNetEML_Adaptive_Preservation_Network_Ensmeble_Machine_Learning.git /content/drive/MyDrive/APNet/APNetCode

fatal: destination path '/content/drive/MyDrive/APNet/APNetCode' already exists and is not an empty directory.


In [4]:
import os
os.chdir('/content/drive/MyDrive/APNet/APNetCode')
!ls -la

total 56
drwx------ 6 root root  4096 Oct 27 07:18 .
drwx------ 3 root root  4096 Oct 27 07:10 ..
drwx------ 3 root root  4096 Oct 27 07:10 apnet
drwx------ 8 root root  4096 Oct 27 07:11 .git
-rw------- 1 root root    88 Oct 27 07:10 .gitignore
drwx------ 2 root root  4096 Oct 27 07:11 images
drwx------ 2 root root  4096 Oct 27 07:18 my_path
-rw------- 1 root root 29799 Oct 27 07:10 README.md
-rw------- 1 root root   186 Oct 27 07:11 requirements.txt
-rw------- 1 root root   385 Oct 27 07:11 setup.py
-rw------- 1 root root   643 Oct 27 07:11 train.py


In [5]:
!pip install tensorflow

In [6]:
import tensorflow as tf
import numpy as np

# Sử dụng Fashion-MNIST
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()

X_train = tf.expand_dims(X_train, axis=-1)
X_test  = tf.expand_dims(X_test, axis=-1)
X_train = tf.image.resize(X_train, (64, 64)).numpy().astype(np.float32)
X_test  = tf.image.resize(X_test, (64, 64)).numpy().astype(np.float32)
X_train = np.repeat(X_train, 3, axis=-1)
X_test  = np.repeat(X_test, 3, axis=-1)

y_train = y_train.flatten()
y_test = y_test.flatten()

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test :", X_test.shape)
print("y_test :", y_test.shape)

X_train: (60000, 64, 64, 3)
y_train: (60000,)
X_test : (10000, 64, 64, 3)
y_test : (10000,)


In [7]:
class CosineLRSchedule(tf.keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, eta_max, eta_min, total_steps):
        super().__init__()
        self.eta_max = eta_max
        self.eta_min = eta_min
        self.total_steps = total_steps

    def __call__(self, step):
        step = tf.cast(step, tf.float32)
        t = tf.minimum(step,tf.cast(self.total_steps, tf.float32))
        lr = (self.eta_min+ 0.5 * (self.eta_max - self.eta_min)* (1.0+ tf.cos((t / self.total_steps) * tf.constant(3.141592653589793))))
        return lr

    def get_config(self):
        return {
            "eta_max": self.eta_max,
            "eta_min": self.eta_min,
            "total_steps": self.total_steps
        }

In [10]:
import tensorflow as tf
from apnet import APNet
ETA_MAX = 1e-5
ETA_MIN = 1e-7
TOTAL_EPOCHS = 10
BATCH_SIZE = 64
STEPS_PER_EPOCH = len(X_train) // BATCH_SIZE
TOTAL_STEPS = TOTAL_EPOCHS * STEPS_PER_EPOCH
lr_schedule = CosineLRSchedule(
    eta_max=ETA_MAX,
    eta_min=ETA_MIN,
    total_steps=TOTAL_STEPS
)
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_accuracy",
    patience=3,
    mode="max",
    restore_best_weights=True,
    verbose=1
)


In [ ]:
model = APNet(
    num_classes=10,
    input_shape=(64, 64, 3),
    embedding_dim=1024,
    backbone=None,
    warmup_epochs=5,
    Frozen=False
)
history = model.fit_dataset(
    train_data=(X_train, y_train),
    val_data=(X_test, y_test),
    epochs = TOTAL_EPOCHS,
    batch_size = BATCH_SIZE,
    learning_rate = lr_schedule,
    save_dir="/content/drive/MyDrive/APNet",
    run_ensemble=True,
    callbacks=[early_stopping]
)

Epoch 1/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 0s 774ms/step - Lce: 1.0012 - Liccl: 1.0587 - Licmrl: 0.0317 - Lpsl: 0.1939 - Luaer: -0.0422 - accuracy: 0.5034 - loss: 7.1868
Epoch 1: val_accuracy improved from None to 0.75786, saving model to /content/drive/MyDrive/APNet/APNet_best.weights.h5

Epoch 1: finished saving model to /content/drive/MyDrive/APNet/APNet_best.weights.h5
938/938 ━━━━━━━━━━━━━━━━━━━━ 752s 791ms/step - Lce: 0.6020 - Liccl: 0.8797 - Licmrl: 0.0160 - Lpsl: 0.1579 - Luaer: -0.0217 - accuracy: 0.6542 - loss: 5.0062 - val_Lce: 0.6248 - val_Liccl: 0.4880 - val_Licmrl: 0.0132 - val_Lpsl: 0.1582 - val_Luaer: -0.0260 - val_accuracy: 0.7579 - val_loss: 3.1336
Epoch 2/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 0s 771ms/step - Lce: 0.5329 - Liccl: 0.8146 - Licmrl: 0.0127 - Lpsl: 0.1527 - Luaer: -0.0272 - accuracy: 0.7913 - loss: 2.8244
Epoch 2: val_accuracy improved from 0.75786 to 0.79628, saving model to /content/drive/MyDrive/APNet/APNet_best.weights.h5

Epoch 2: finished saving model to /co